# Import Libraries

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/Pawnder/colab_setup.py"

In [ ]:
import os
import sys
import glob
import json
import shutil
import re
import random
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D,
    Dense, Dropout, Flatten, Concatenate, BatchNormalization,
    TimeDistributed, LSTM, Bidirectional
)
from tensorflow.keras.applications import (
    MobileNetV2, ResNet50, EfficientNetB0
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2
import yaml
from collections import defaultdict
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from ml.notebooks.split_dataset import split_dataset

In [ ]:
# Install required packages if not already installed
try:
    import cv2
except ImportError:
    !pip install opencv-python-headless

try:
    from tqdm import tqdm
except ImportError:
    !pip install tqdm

# Part 1: Configuration and Setup

In [ ]:
# Project paths setup
project_root = '/content/drive/MyDrive/Colab Notebooks/Pawnder'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Try to import from config
try:
    from config import DATA_DIRS, ensure_directories, EMOTION_MAPPING
except ModuleNotFoundError:
    print("Could not import config module. Adjusting path...")
    # Try with full path
    sys.path.insert(0, os.path.abspath(project_root))
    try:
        from config import DATA_DIRS, ensure_directories, EMOTION_MAPPING
    except ModuleNotFoundError:
        # Define default configuration if import fails
        print("Creating default configuration")

        def normalize_path(path):
            """Normalize a path to avoid duplicates caused by different representations"""
            if isinstance(path, str) and path.startswith('s3://'):
                return path  # Don't normalize S3 paths
            return os.path.normpath(path)

        # Default data directories
        base_dir = "/content/drive/MyDrive/Colab Notebooks/Pawnder"
        output_dir = os.path.join(base_dir, "Data/processed")
        os.makedirs(output_dir, exist_ok=True)
        DATA_ROOT = normalize_path(os.path.join(base_dir, 'Data'))

        DATA_DIRS = {
            'raw': os.path.join(DATA_ROOT, 'Raw'),
            'stanford_original': os.path.join(DATA_ROOT, 'Raw', 'stanford_dog_pose'),
            'personal_dataset': os.path.join(DATA_ROOT, 'Raw', 'personal_dataset'),
            'matrix': os.path.join(DATA_ROOT, 'Matrix'),
            'interim': os.path.join(DATA_ROOT, 'interim'),
            'processed': os.path.join(DATA_ROOT, 'processed'),
            'models': os.path.join(base_dir, 'Models'),
        }

        # Default emotion mapping
        EMOTION_MAPPING = {
            "Happy or Playful": "Happy/Playful",
            "Relaxed": "Relaxed",
            "Submissive": "Submissive/Appeasement",
            "Excited": "Happy/Playful",
            "Drowsy or Bored": "Relaxed",
            "Curious or Confused": "Curiosity/Alertness",
            "Confident or Alert": "Curiosity/Alertness",
            "Jealous": "Stressed",
            "Stressed": "Stressed",
            "Frustrated": "Stressed",
            "Unsure or Uneasy": "Fearful/Anxious",
            "Possessive, Territorial, Dominant": "Aggressive/Threatening",
            "Fear or Aggression": "Fearful/Anxious",
            "Pain": "Stressed"
        }

        def ensure_directories():
            """Ensure all required directories exist"""
            for path in DATA_DIRS.values():
                if isinstance(path, str) and not path.startswith('s3://'):
                    Path(path).mkdir(parents=True, exist_ok=True)

            # Create subdirectories for processed data
            for split in ['train', 'validation', 'test']:
                split_path = os.path.join(DATA_DIRS['processed'], split)
                for subdir in ['images', 'annotations']:
                    path = os.path.join(split_path, subdir)
                    Path(path).mkdir(parents=True, exist_ok=True)

# Make sure directories exist
ensure_directories()

# Define configuration for the model (can be loaded from YAML if available)
def load_config(config_path="config.yaml"):
    """Load configuration from YAML file or use defaults"""
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        return config
    else:
        # Use default configuration
        print(f"Config file not found: {config_path}, using defaults")
        default_config = {
            "data": {
                "base_dir": project_root,
                "raw_data_dir": os.path.join(DATA_DIRS['raw']),
                "processed_data_dir": os.path.join(DATA_DIRS['processed']),
                "train_split": 0.7,
                "val_split": 0.15,
                "test_split": 0.15
            },
            "model": {
                "image_size": [224, 224, 3],
                "batch_size": 32,
                "learning_rate": 0.001,
                "dropout_rate": 0.5,
                "backbone": "mobilenetv2",
                "early_stopping_patience": 5
            },
            "training": {
                "checkpoint_dir": os.path.join(DATA_DIRS['models'], "checkpoints"),
                "logs_dir": os.path.join(DATA_DIRS['models'], "logs")
            },
            "inference": {
                "confidence_threshold": 0.6,
                "behavior_threshold": 0.5,
                "output_dir": os.path.join(DATA_DIRS['processed'], "predictions")
            }
        }

        # Save default config for future use
        os.makedirs(os.path.dirname(config_path), exist_ok=True)
        with open(config_path, 'w') as f:
            yaml.dump(default_config, f)

        return default_config

def load_config(config_path="/content/drive/MyDrive/Colab Notebooks/Pawnder/config.yaml"):
    try:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        print(f"Successfully loaded config from {config_path}")
        return config
    except FileNotFoundError:
        print(f"Config file not found at: {config_path}")
        # Use the default config creation logic from the previous function
        return create_default_config(config_path)
    except PermissionError:
        print(f"Permission denied when trying to access: {config_path}")
        return create_default_config(config_path)
    except Exception as e:
        print(f"An error occurred when loading config: {str(e)}")
        return create_default_config(config_path)

# Define emotion classes from the EMOTION_MAPPING
CLASS_NAMES = list(set(EMOTION_MAPPING.values()))
# Safe class names for directories
SAFE_CLASS_NAMES = [emotion.replace("/", "_").replace("\\", "_") for emotion in CLASS_NAMES]
CLASS_MAP = dict(zip(CLASS_NAMES, SAFE_CLASS_NAMES))

# PART 2: Data Processing Pipeline

In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/Pawnder/scripts/process_dataset.py"
# --stanford_only --force_reprocess
# !python process_dataset.py --force_reprocess
# !python process_dataset.py --only_splits

In [ ]:
!python "/content/drive/MyDrive/Colab Notebooks/Pawnder/scripts/stanford_keypoint_processor.py"

In [ ]:

# Keypoint Updater
import os
import json
import pandas as pd
import numpy as np
import glob
from tqdm import tqdm
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("KeypointUpdater")

def load_keypoint_annotations(keypoint_dir):
    """Load keypoint annotations from files"""
    logger.info(f"Loading keypoint annotations from {keypoint_dir}...")

    if not os.path.exists(keypoint_dir):
        logger.warning(f"Keypoint directory not found: {keypoint_dir}")
        return {}

    # Load all keypoint files
    keypoints = {}
    keypoint_files = glob.glob(os.path.join(keypoint_dir, "*.json"))

    for kp_file in tqdm(keypoint_files, desc="Loading keypoint files"):
        try:
            with open(kp_file, 'r') as f:
                keypoint_data = json.load(f)

            # Extract the image ID from filename
            filename = os.path.basename(kp_file)
            image_id = os.path.splitext(filename)[0]

            # Store keypoints indexed by image ID
            keypoints[image_id] = keypoint_data

        except Exception as e:
            logger.warning(f"Error loading keypoint file {kp_file}: {str(e)}")

    logger.info(f"Loaded keypoints for {len(keypoints)} images")
    return keypoints

def update_annotations_with_keypoints(annotations_path, keypoints, output_path=None):
    """Update annotations CSV with keypoint data"""
    logger.info(f"Updating annotations at {annotations_path} with keypoints...")

    # Load annotations
    try:
        annotations = pd.read_csv(annotations_path)
    except Exception as e:
        logger.error(f"Error loading annotations: {e}")
        return None

    # Add keypoint columns
    annotations['has_keypoints'] = False
    annotations['keypoint_file'] = None

    # Track matches
    match_count = 0

    # Update each row
    for idx, row in tqdm(annotations.iterrows(), total=len(annotations), desc="Matching keypoints"):
        # Get image path
        if 'image_path' not in row:
            continue

        image_path = row['image_path']

        # Try different naming patterns to match with keypoints
        image_id = os.path.splitext(os.path.basename(image_path))[0]

        # Direct match
        if image_id in keypoints:
            annotations.at[idx, 'has_keypoints'] = True
            annotations.at[idx, 'keypoint_file'] = f"{image_id}.json"
            match_count += 1
            continue

        # Try with 'stanford_' prefix removed (if present)
        if image_id.startswith('stanford_'):
            stanford_id = image_id[9:]  # Remove 'stanford_' prefix
            if stanford_id in keypoints:
                annotations.at[idx, 'has_keypoints'] = True
                annotations.at[idx, 'keypoint_file'] = f"{stanford_id}.json"
                match_count += 1
                continue

        # Try with original ID
        if 'original_id' in row and not pd.isna(row['original_id']):
            original_id = os.path.splitext(os.path.basename(row['original_id']))[0]
            if original_id in keypoints:
                annotations.at[idx, 'has_keypoints'] = True
                annotations.at[idx, 'keypoint_file'] = f"{original_id}.json"
                match_count += 1
                continue

    logger.info(f"Matched keypoints for {match_count} out of {len(annotations)} annotations ({match_count/len(annotations)*100:.1f}%)")

    # Save updated annotations if output path provided
    if output_path:
        annotations.to_csv(output_path, index=False)
        logger.info(f"Saved updated annotations to {output_path}")

    return annotations

# Define your directories explicitly
base_dir = "/content/drive/MyDrive/Colab Notebooks/Pawnder"
keypoint_dir = f"{base_dir}/Data/annotations/stanford_keypoints"
# Set to the processed directory which contains train, validation, test folders
annotations_dir = f"{base_dir}/Data/processed"

# Verify directories exist
if not os.path.exists(keypoint_dir):
    logger.error(f"Keypoint directory not found: {keypoint_dir}")
else:
    # Load keypoints
    keypoints = load_keypoint_annotations(keypoint_dir)

    if not keypoints:
        logger.error("No keypoints loaded. Exiting.")
    else:
        # Update annotations for each split
        for split in ['train', 'val', 'test']:
            split_dir = os.path.join(annotations_dir, split)
            if not os.path.exists(split_dir):
                logger.warning(f"Split directory not found: {split_dir}")
                continue

            annotations_path = os.path.join(split_dir, 'annotations.csv')
            if not os.path.exists(annotations_path):
                logger.warning(f"Annotations file not found: {annotations_path}")
                continue

            # Update annotations
            updated = update_annotations_with_keypoints(
                annotations_path,
                keypoints,
                output_path=annotations_path  # Overwrite original
            )

            if updated is not None:
                logger.info(f"Successfully updated {split} annotations")

# Part 3: Model Architecture

# Part 4: Data Generator and Training

In [ ]:
test

In [ ]:
# Import the fixed model
from fix_path_dog_emotion import DogEmotionWithBehaviors, find_directory

# Try to auto-detect the correct directory
train_dir = find_directory()
if train_dir:
    # Use the parent directory as the processed_dir
    processed_dir = os.path.dirname(train_dir)
    # Create classifier with the found directory
    classifier = DogEmotionWithBehaviors(base_dir=os.path.dirname(processed_dir))
else:
    # Use default directory
    classifier = DogEmotionWithBehaviors()

# Train with fewer epochs for testing
history, model_dir = classifier.train(
    epochs=10,               # Start with fewer epochs to test
    batch_size=32,
    img_size=(224, 224),
    fine_tune=False          # Start without fine-tuning for faster testing
)

# Part 5: Inference and Visualization

In [ ]:
def predict_image(model, image_path, config, class_names=None, show_visualization=True):
    """
    Predict emotion from a single image
    
    Args:
        model (tf.keras.Model): Trained model
        image_path (str): Path to the image
        config (dict): Configuration dictionary
        class_names (list): List of emotion class names
        show_visualization (bool): Whether to display visualization
        
    Returns:
        dict: Prediction results
    """
    if class_names is None:
        class_names = CLASS_NAMES
        
    img_size = tuple(config['model']['image_size'][:2])
    
    # Load and preprocess image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not load image: {image_path}")
    
    # Convert BGR to RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Preprocess image
    img_resized = cv2.resize(img_rgb, (img_size[1], img_size[0]))
    img_norm = img_resized.astype(np.float32) / 255.0
    
    # Create dummy behavior input (all zeros for now)
    behavior_size = model.inputs[1].shape[1]
    behavior_input = np.zeros((1, behavior_size), dtype=np.float32)
    
    # Make prediction
    inputs = {
        'image_input': np.expand_dims(img_norm, axis=0),
        'behavior_input': behavior_input
    }
    
    predictions = model.predict(inputs, verbose=0)
    emotion_probs, confidence = predictions
    
    # Get the predicted emotion
    emotion_idx = np.argmax(emotion_probs[0])
    emotion = class_names[emotion_idx]
    emotion_score = float(emotion_probs[0][emotion_idx])
    confidence_score = float(confidence[0][0])
    
    # Create result dictionary
    result = {
        'emotion': emotion,
        'emotion_score': emotion_score,
        'confidence': confidence_score,
        'all_emotions': {class_names[i]: float(emotion_probs[0][i]) for i in range(len(class_names))}
    }
    
    # Visualize result if requested
    if show_visualization:
        visualize_prediction(img_rgb, result, image_path)
    
    return result

def visualize_prediction(img, result, image_path, save_path=None):
    """
    Visualize prediction result
    
    Args:
        img (np.ndarray): Image array in RGB format
        result (dict): Prediction result
        image_path (str): Path to the original image
        save_path (str): Path to save visualization (optional)
    """
    # Create figure
    plt.figure(figsize=(12, 6))
    
    # Display image
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f"Predicted: {result['emotion']} ({result['emotion_score']:.2f})")
    plt.axis('off')
    
    # Plot emotion probabilities
    plt.subplot(1, 2, 2)
    emotions = list(result['all_emotions'].keys())
    scores = list(result['all_emotions'].values())
    
    # Sort by score
    sorted_indices = np.argsort(scores)[::-1]
    emotions = [emotions[i] for i in sorted_indices]
    scores = [scores[i] for i in sorted_indices]
    
    # Bar chart for top 5 emotions
    bars = plt.barh(emotions[:5], scores[:5], color='skyblue')
    
    # Add confidence score
    plt.axvline(x=result['confidence'], color='red', linestyle='--', label=f'Confidence: {result["confidence"]:.2f}')
    
    # Highlight predicted emotion
    for i, e in enumerate(emotions[:5]):
        if e == result['emotion']:
            bars[i].set_color('green')
    
    plt.xlabel('Score')
    plt.ylabel('Emotion')
    plt.title('Top 5 Emotion Predictions')
    plt.xlim(0, 1)
    plt.legend()
    plt.tight_layout()
    
    # Save if requested
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Visualization saved to {save_path}")
    
    plt.show()

def process_video(model, video_path, config, class_names=None, output_path=None, frame_interval=5):
    """
    Process a video and predict emotions
    
    Args:
        model (tf.keras.Model): Trained model
        video_path (str): Path to the video
        config (dict): Configuration dictionary
        class_names (list): List of emotion class names
        output_path (str): Path to save output video (optional)
        frame_interval (int): Process every Nth frame
        
    Returns:
        dict: Prediction results over time
    """
    if class_names is None:
        class_names = CLASS_NAMES
        
    img_size = tuple(config['model']['image_size'][:2])
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")
    
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Setup output video if requested
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(
            output_path,
            fourcc,
            fps / frame_interval,  # Adjust output FPS based on frame interval
            (frame_width, frame_height)
        )
    
    # Create dummy behavior input (all zeros for now)
    behavior_size = model.inputs[1].shape[1]
    behavior_input = np.zeros((1, behavior_size), dtype=np.float32)
    
    # Process video frames
    results = {
        'video_path': video_path,
        'fps': fps,
        'total_frames': total_frames,
        'frames_analyzed': 0,
        'emotion_timeline': [],
        'dominant_emotion': None
    }
    
    frame_count = 0
    emotion_counts = {emotion: 0 for emotion in class_names}
    
    print(f"Processing video: {video_path}")
    
    try:
        with tqdm(total=total_frames) as pbar:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                
                # Process every Nth frame
                if frame_count % frame_interval == 0:
                    # Convert to RGB
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    
                    # Preprocess image
                    img_resized = cv2.resize(frame_rgb, (img_size[1], img_size[0]))
                    img_norm = img_resized.astype(np.float32) / 255.0
                    
                    # Make prediction
                    inputs = {
                        'image_input': np.expand_dims(img_norm, axis=0),
                        'behavior_input': behavior_input
                    }
                    
                    predictions = model.predict(inputs, verbose=0)
                    emotion_probs, confidence = predictions
                    
                    # Get the predicted emotion
                    emotion_idx = np.argmax(emotion_probs[0])
                    emotion = class_names[emotion_idx]
                    emotion_score = float(emotion_probs[0][emotion_idx])
                    confidence_score = float(confidence[0][0])
                    
                    # Count emotions for dominant emotion calculation
                    emotion_counts[emotion] += 1
                    
                    # Add to timeline
                    results['emotion_timeline'].append({
                        'frame': frame_count,
                        'time': frame_count / fps,
                        'emotion': emotion,
                        'emotion_score': emotion_score,
                        'confidence': confidence_score
                    })
                    
                    # Update count of analyzed frames
                    results['frames_analyzed'] += 1
                    
                    # Draw results on frame for output video
                    if output_path:
                        # Add text
                        text = f"{emotion}: {emotion_score:.2f}, Conf: {confidence_score:.2f}"
                        cv2.putText(
                            frame, text, (10, 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2
                        )
                        
                        # Write frame to output video
                        out.write(frame)
                
                frame_count += 1
                pbar.update(1)
    
    finally:
        # Release resources
        cap.release()
        if output_path and 'out' in locals():
            out.release()
    
    # Calculate dominant emotion
    if results['frames_analyzed'] > 0:
        dominant_emotion = max(emotion_counts.items(), key=lambda x: x[1])[0]
        results['dominant_emotion'] = dominant_emotion
        
        # Calculate emotion distribution
        results['emotion_distribution'] = {
            emotion: count / results['frames_analyzed'] if results['frames_analyzed'] > 0 else 0
            for emotion, count in emotion_counts.items()
        }
    
    # Visualize emotion over time
    if results['frames_analyzed'] > 0:
        visualize_video_emotions(results, class_names, output_path)
    
    return results

def visualize_video_emotions(results, class_names, output_path=None):
    """
    Visualize emotions over time in a video
    
    Args:
        results (dict): Video processing results
        class_names (list): List of emotion class names
        output_path (str): Path to save visualization (optional)
    """
    # Create figure
    plt.figure(figsize=(14, 10))
    
    # Plot emotion distribution
    plt.subplot(2, 1, 1)
    emotions = []
    percentages = []
    
    for emotion, percent in sorted(results['emotion_distribution'].items(), key=lambda x: x[1], reverse=True):
        if percent > 0:
            emotions.append(emotion)
            percentages.append(percent)
    
    bars = plt.bar(emotions, percentages, color='skyblue')
    
    # Highlight dominant emotion
    for i, emotion in enumerate(emotions):
        if emotion == results['dominant_emotion']:
            bars[i].set_color('green')
    
    plt.xlabel('Emotion')
    plt.ylabel('Percentage')
    plt.title('Emotion Distribution')
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha='right')
    
    # Plot emotion timeline
    plt.subplot(2, 1, 2)
    
    # Convert emotions to numeric values
    emotion_to_idx = {emotion: i for i, emotion in enumerate(class_names)}
    times = [entry['time'] for entry in results['emotion_timeline']]
    emotion_values = [emotion_to_idx[entry['emotion']] for entry in results['emotion_timeline']]
    
    plt.scatter(times, emotion_values, alpha=0.6)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Emotion')
    plt.title('Emotion Timeline')
    plt.yticks(range(len(class_names)), class_names)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save if requested
    if output_path:
        # Create a path for the visualization by modifying the output video path
        vis_path = os.path.splitext(output_path)[0] + "_emotions.png"
        plt.savefig(vis_path, dpi=300, bbox_inches='tight')
        print(f"Emotion visualization saved to {vis_path}")
    
    plt.show()

In [ ]:
# Save the fixed model code
with open('fix_path_dog_emotion.py', 'w') as f:
    """
Dog Emotion Model with Behavioral Feature Integration - Fixed Path Handling

This implementation focuses on integrating the Primary Behavior Matrix
features while maintaining compatibility with TensorFlow's recent versions.
"""

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
from datetime import datetime
import cv2
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report

# Try to configure GPU settings
try:
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Found {len(gpus)} GPU(s). Memory growth enabled")
        
    # Try to enable mixed precision
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision enabled")
except Exception as e:
    print(f"GPU configuration error: {str(e)}")

class DogEmotionWithBehaviors:
    """
    Dog emotion classification model with behavioral feature integration
    """
    
    def __init__(self, base_dir="/content/drive/MyDrive/Colab Notebooks/Pawnder"):
        """
        Initialize the classifier
        
        Args:
            base_dir: Base directory containing the project
        """
        self.base_dir = base_dir
        self.processed_dir = os.path.join(base_dir, "Data/processed")
        self.model = None
        self.class_names = []
        self.behavior_columns = []
        self.model_dir = os.path.join(base_dir, "Models")
        
        # Create models directory if it doesn't exist
        os.makedirs(self.model_dir, exist_ok=True)
        
        # Print paths for debugging
        print(f"Base directory: {self.base_dir}")
        print(f"Processed directory: {self.processed_dir}")
        print(f"Model directory: {self.model_dir}")
        
        # Verify directories
        if not os.path.exists(self.processed_dir):
            print(f"Warning: Processed directory not found at {self.processed_dir}")
        
        if not os.path.exists(os.path.join(self.processed_dir, 'train_by_class')):
            print(f"Warning: Train directory not found at {os.path.join(self.processed_dir, 'train_by_class')}")
            # Try to look for alternative locations
            possible_dirs = [
                os.path.join(base_dir, "data/processed/train_by_class"),
                os.path.join(base_dir, "Data/processed/train_by_class"),
                os.path.join("/content/drive/MyDrive/Colab Notebooks/Pawnder/Data/processed/train_by_class")
            ]
            for possible_dir in possible_dirs:
                if os.path.exists(possible_dir):
                    print(f"Found alternative train directory: {possible_dir}")
                    # Update processed dir accordingly
                    self.processed_dir = os.path.dirname(possible_dir)
                    print(f"Updated processed directory to: {self.processed_dir}")
                    break
    
    def create_model(self, num_classes, behavior_size=10, img_size=(224, 224, 3)):
        """
        Create model with both image and behavioral inputs
        
        Args:
            num_classes: Number of emotion classes
            behavior_size: Number of behavioral features
            img_size: Input image size (height, width, channels)
            
        Returns:
            Compiled Keras model
        """
        # Create image input branch
        image_input = tf.keras.Input(shape=img_size, name='image_input')
        
        # Use MobileNetV2 as base model
        base_model = applications.MobileNetV2(
            input_shape=img_size,
            include_top=False,
            weights='imagenet'
        )
        
        # Freeze base model
        base_model.trainable = False
        
        # Extract features from image
        x = base_model(image_input)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.3)(x)
        
        # Create behavioral input branch
        behavior_input = tf.keras.Input(shape=(behavior_size,), name='behavior_input')
        b = layers.Dense(64, activation='relu')(behavior_input)
        b = layers.BatchNormalization()(b)
        b = layers.Dropout(0.3)(b)
        
        # Combine image and behavior branches
        combined = layers.Concatenate()([x, b])
        
        # Classification head
        combined = layers.Dense(128, activation='relu')(combined)
        combined = layers.Dropout(0.3)(combined)
        output = layers.Dense(num_classes, activation='softmax')(combined)
        
        # Create and compile model
        model = tf.keras.Model(
            inputs={'image_input': image_input, 'behavior_input': behavior_input},
            outputs=output
        )
        
        # Compile model
        model.compile(
            optimizer=tf.keras.optimizers.Adam(1e-4),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # Store model
        self.model = model
        return model
    
    def fine_tune_model(self, unfreeze_layers=15):
        """
        Fine-tune the model by unfreezing some base model layers
        
        Args:
            unfreeze_layers: Number of layers to unfreeze from the end
            
        Returns:
            Fine-tuned model
        """
        if self.model is None:
            raise ValueError("No model to fine-tune. Create a model first.")
        
        # Find the base model
        base_model = None
        for layer in self.model.layers:
            if isinstance(layer, tf.keras.Model):
                base_model = layer
                break
        
        if base_model is None:
            print("Base model not found. Looking deeper in the model structure...")
            # Try to find it in the functional API structure
            for layer in self.model.layers:
                if hasattr(layer, 'layer') and isinstance(layer.layer, tf.keras.Model):
                    base_model = layer.layer
                    break
        
        if base_model is None:
            print("Could not find base model for fine-tuning")
            return self.model
        
        # Make base model trainable
        base_model.trainable = True
        
        # Freeze all layers except the last few
        for layer in base_model.layers[:-unfreeze_layers]:
            layer.trainable = False
        
        # Recompile with lower learning rate
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(1e-5),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        print(f"Model fine-tuned: Unfroze last {unfreeze_layers} layers of the base model")
        return self.model
    
    def load_class_data(self, split_name='train'):
        """
        Load image paths and labels from class directories
        
        Args:
            split_name: Data split ('train', 'validation', 'test')
            
        Returns:
            tuple: (image_paths, labels, class_names)
        """
        # Get split directory
        if split_name == 'train':
            split_dir = os.path.join(self.processed_dir, 'train_by_class')
        elif split_name == 'validation':
            split_dir = os.path.join(self.processed_dir, 'val_by_class')
        elif split_name == 'test':
            split_dir = os.path.join(self.processed_dir, 'test_by_class')
        else:
            raise ValueError(f"Invalid split name: {split_name}")
        
        # Check if directory exists
        if not os.path.exists(split_dir):
            raise FileNotFoundError(f"Split directory not found: {split_dir}")
        
        # Get class directories
        class_dirs = [d for d in os.listdir(split_dir) 
                     if os.path.isdir(os.path.join(split_dir, d)) and d != 'unknown']
        
        # Sort for consistent order
        class_dirs.sort()
        
        # Save class names
        self.class_names = class_dirs
        
        # Prepare lists for images and labels
        image_paths = []
        labels = []
        absolute_paths = []  # Store absolute paths for debugging
        
        # Load images and labels
        print(f"Loading {split_name} data from {split_dir}")
        for class_idx, class_name in enumerate(class_dirs):
            class_dir = os.path.join(split_dir, class_name)
            image_files = [f for f in os.listdir(class_dir) 
                          if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            
            print(f"  {class_name}: {len(image_files)} images")
            
            # Add images and labels
            for img_file in image_files:
                # Store only the full absolute path
                abs_path = os.path.abspath(os.path.join(class_dir, img_file))
                image_paths.append(abs_path)
                absolute_paths.append(abs_path)
                labels.append(class_idx)
        
        # Save a few paths for debugging
        if len(absolute_paths) > 0:
            print(f"Example paths:")
            for i in range(min(3, len(absolute_paths))):
                print(f"  {absolute_paths[i]}")
                # Check if file exists
                print(f"    Exists: {os.path.exists(absolute_paths[i])}")
        
        return image_paths, labels, class_dirs
    
    def load_behavior_data(self):
        """
        Load behavioral data from Primary Behavior Matrix
        
        Returns:
            tuple: (behavior_data, behavior_columns)
        """
        # Try to find the primary matrix data
        matrix_paths = [
            os.path.join(self.base_dir, "Data/Matrix/primary_behavior_matrix.json"),
            os.path.join(self.base_dir, "Data/Matrix/Primary Behavior Matrix.xlsx"),
            os.path.join(self.base_dir, "data/Matrix/primary_behavior_matrix.json")
        ]
        
        behavior_data = {}
        behavior_columns = []
        
        # Try to load the matrix from any of the possible paths
        loaded = False
        
        # First try to load from annotations
        annotations_path = os.path.join(self.processed_dir, "combined_annotations.json")
        if os.path.exists(annotations_path):
            try:
                print(f"Loading annotations from {annotations_path}")
                with open(annotations_path, 'r') as f:
                    annotations = json.load(f)
                
                # Extract behavior columns
                first_item = next(iter(annotations.values()))
                behavior_columns = [k for k in first_item.keys() if k.startswith('behavior_')]
                
                if behavior_columns:
                    print(f"Found {len(behavior_columns)} behavior columns in annotations")
                    
                    # Create behavior data dictionary
                    for image_id, data in annotations.items():
                        # Extract behaviors
                        behaviors = []
                        for col in behavior_columns:
                            if col in data:
                                value = data[col]
                                # Convert to float
                                if isinstance(value, bool):
                                    value = 1.0 if value else 0.0
                                elif isinstance(value, (int, float)):
                                    value = float(value)
                                else:
                                    value = 0.0
                                behaviors.append(value)
                            else:
                                behaviors.append(0.0)
                        
                        # Store behaviors by full path and basename for easier matching
                        behavior_data[image_id] = behaviors
                        # Also store by basename for easier matching
                        basename = os.path.basename(image_id)
                        if basename:
                            behavior_data[basename] = behaviors
                    
                    loaded = True
                    print(f"Loaded behavior data for {len(behavior_data)} images")
            except Exception as e:
                print(f"Error loading behaviors from annotations: {str(e)}")
        
        # Try to load the Primary Behavior Matrix directly
        if not loaded:
            for matrix_path in matrix_paths:
                if os.path.exists(matrix_path):
                    try:
                        print(f"Trying to load matrix from {matrix_path}")
                        # Handle JSON format
                        if matrix_path.endswith('.json'):
                            with open(matrix_path, 'r') as f:
                                matrix_data = json.load(f)
                            
                            # Extract behavior columns
                            behavior_columns = [k for k in matrix_data.keys() if k.startswith('behavior_')]
                            
                            if behavior_columns:
                                print(f"Found {len(behavior_columns)} behavior columns in matrix")
                                loaded = True
                                break
                        
                        # Handle Excel format (requires pandas)
                        elif matrix_path.endswith('.xlsx'):
                            import pandas as pd
                            matrix_df = pd.read_excel(matrix_path)
                            
                            # Extract behavior columns
                            behavior_columns = [col for col in matrix_df.columns if col.startswith('behavior_')]
                            
                            if behavior_columns:
                                print(f"Found {len(behavior_columns)} behavior columns in Excel matrix")
                                loaded = True
                                break
                    except Exception as e:
                        print(f"Error loading matrix from {matrix_path}: {str(e)}")
        
        # If no behavior data found, create dummy data
        if not behavior_data:
            print("No behavior data found. Creating placeholder behavior data.")
            behavior_columns = ["behavior_dummy"]
            behavior_data = {}  # Will be filled with zeros during batching
        
        self.behavior_columns = behavior_columns
        return behavior_data, behavior_columns
    
    def create_data_generator(self, 
                              image_paths, 
                              labels, 
                              behavior_data=None,
                              img_size=(224, 224), 
                              batch_size=32, 
                              augment=False):
        """
        Create a data generator for training/validation
        
        Args:
            image_paths: List of image paths
            labels: List of class indices
            behavior_data: Dictionary mapping image paths to behavior features
            img_size: Image dimensions (height, width)
            batch_size: Batch size
            augment: Whether to apply data augmentation
            
        Returns:
            Data generator
        """
        # Convert labels to one-hot encoding
        num_classes = len(set(labels))
        labels_onehot = tf.keras.utils.to_categorical(labels, num_classes)
        
        # Get behavior size
        if behavior_data and len(behavior_data) > 0:
            first_key = next(iter(behavior_data.keys()))
            behavior_size = len(behavior_data[first_key])
        else:
            behavior_size = len(self.behavior_columns) if self.behavior_columns else 1
        
        # Create data generator class
        class DataGenerator(tf.keras.utils.Sequence):
            def __init__(self, image_paths, labels, behavior_data, behavior_size, img_size, batch_size, augment):
                self.image_paths = image_paths
                self.labels = labels
                self.behavior_data = behavior_data
                self.behavior_size = behavior_size
                self.img_size = img_size
                self.batch_size = batch_size
                self.augment = augment
                self.indices = np.arange(len(self.image_paths))
                
                # Setup augmentation if needed
                if self.augment:
                    self.img_gen = ImageDataGenerator(
                        rotation_range=20,
                        width_shift_range=0.2,
                        height_shift_range=0.2,
                        shear_range=0.1,
                        zoom_range=0.2,
                        horizontal_flip=True,
                        brightness_range=[0.8, 1.2],
                        fill_mode='nearest'
                    )
            
            def __len__(self):
                """Number of batches per epoch"""
                return int(np.ceil(len(self.image_paths) / self.batch_size))
            
            def __getitem__(self, idx):
                """Generate one batch of data"""
                batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
                batch_size = len(batch_indices)
                
                # Initialize batch arrays
                batch_images = np.zeros((batch_size, *self.img_size, 3), dtype=np.float32)
                batch_behaviors = np.zeros((batch_size, self.behavior_size), dtype=np.float32)
                batch_labels = np.zeros((batch_size, num_classes), dtype=np.float32)
                
                # Fill batch data
                for i, idx in enumerate(batch_indices):
                    # Get image path - already absolute from load_class_data
                    img_path = self.image_paths[idx]
                    
                    try:
                        # Load and preprocess image
                        img = cv2.imread(img_path)
                        if img is None:
                            raise ValueError(f"Failed to load image: {img_path}")
                        
                        # Convert BGR to RGB
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        
                        # Resize image
                        img = cv2.resize(img, self.img_size)
                        
                        # Normalize pixel values
                        img = img.astype(np.float32) / 255.0
                        
                        # Apply augmentation if enabled
                        if self.augment:
                            img = self.img_gen.random_transform(img)
                        
                        # Store image and label
                        batch_images[i] = img
                        batch_labels[i] = self.labels[idx]
                        
                        # Add behavioral data if available
                        if self.behavior_data:
                            # Try a few different ways to match the behavior data
                            matched = False
                            
                            # 1. Try by full path
                            if img_path in self.behavior_data:
                                batch_behaviors[i] = self.behavior_data[img_path]
                                matched = True
                            
                            # 2. Try by basename
                            if not matched:
                                basename = os.path.basename(img_path)
                                if basename in self.behavior_data:
                                    batch_behaviors[i] = self.behavior_data[basename]
                                    matched = True
                            
                            # 3. Try by basename without extension
                            if not matched:
                                basename_no_ext = os.path.splitext(os.path.basename(img_path))[0]
                                if basename_no_ext in self.behavior_data:
                                    batch_behaviors[i] = self.behavior_data[basename_no_ext]
                                    matched = True
                            
                            # If still no match, leave as zeros (already initialized)
                    
                    except Exception as e:
                        print(f"Error processing {img_path}: {str(e)}")
                        # Use zeros for failed images (already initialized)
                
                # Return the batch (using the proper input format for model)
                inputs = {
                    'image_input': batch_images,
                    'behavior_input': batch_behaviors
                }
                
                return inputs, batch_labels
            
            def on_epoch_end(self):
                """Shuffle indices after each epoch"""
                if self.augment:  # Only shuffle training data
                    np.random.shuffle(self.indices)
        
        # Create and return data generator
        return DataGenerator(image_paths, labels_onehot, behavior_data, behavior_size, 
                            img_size, batch_size, augment)
    
    def train(self, epochs=50, batch_size=32, img_size=(224, 224), fine_tune=True):
        """
        Train the model
        
        Args:
            epochs: Number of training epochs
            batch_size: Batch size
            img_size: Image dimensions (height, width)
            fine_tune: Whether to perform fine-tuning
            
        Returns:
            tuple: (history, model_dir)
        """
        # Load data
        train_paths, train_labels, class_names = self.load_class_data('train')
        val_paths, val_labels, _ = self.load_class_data('validation')
        
        # Load behavior data
        behavior_data, behavior_columns = self.load_behavior_data()
        
        # Create data generators
        train_gen = self.create_data_generator(
            train_paths, train_labels, behavior_data, img_size, batch_size, augment=True)
        
        val_gen = self.create_data_generator(
            val_paths, val_labels, behavior_data, img_size, batch_size, augment=False)
        
        # Create model if not already created
        if self.model is None:
            self.create_model(
                num_classes=len(class_names),
                behavior_size=len(behavior_columns),
                img_size=(*img_size, 3)
            )
        
        # Create model directory with timestamp
        timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
        model_dir = os.path.join(self.model_dir, f"dog_emotion_{timestamp}")
        os.makedirs(model_dir, exist_ok=True)
        
        # Create callbacks
        callbacks = [
            # Early stopping
            EarlyStopping(
                monitor='val_accuracy',
                patience=10,
                restore_best_weights=True
            ),
            # Model checkpoints
            ModelCheckpoint(
                os.path.join(model_dir, 'best_model.h5'),
                monitor='val_accuracy',
                save_best_only=True,
                mode='max'
            ),
            # Learning rate reduction
            ReduceLROnPlateau(
                monitor='val_accuracy',
                factor=0.5,
                patience=5,
                min_lr=1e-6,
                mode='max'
            )
        ]
        
        # Train model
        print(f"Training model with {len(train_paths)} images and {len(behavior_columns)} behavior features")
        history = self.model.fit(
            train_gen,
            epochs=epochs,
            validation_data=val_gen,
            callbacks=callbacks
        )
        
        # Fine-tuning if requested
        if fine_tune:
            print("Fine-tuning the model...")
            
            # Fine-tune the model
            self.fine_tune_model(unfreeze_layers=15)
            
            # Train with fine-tuning
            ft_history = self.model.fit(
                train_gen,
                epochs=20,  # Fewer epochs for fine-tuning
                validation_data=val_gen,
                callbacks=callbacks
            )
            
            # Extend history
            for k in history.history:
                if k in ft_history.history:
                    history.history[k].extend(ft_history.history[k])
        
        # Save final model
        self.model.save(os.path.join(model_dir, 'final_model.h5'))
        
        # Save class names and behavior columns
        with open(os.path.join(model_dir, 'model_metadata.json'), 'w') as f:
            json.dump({
                'class_names': class_names,
                'behavior_columns': behavior_columns,
                'img_size': img_size,
                'training_timestamp': timestamp
            }, f, indent=2)
        
        # Save training history
        history_dict = {}
        for k, v in history.history.items():
            history_dict[k] = [float(x) for x in v]
            
        with open(os.path.join(model_dir, 'training_history.json'), 'w') as f:
            json.dump(history_dict, f, indent=2)
        
        # Plot training history
        self.plot_training_history(history, model_dir)
        
        # Evaluate on test set
        self.evaluate(model_dir)
        
        return history, model_dir
    
    def evaluate(self, model_dir):
        """
        Evaluate the model on the test set
        
        Args:
            model_dir: Directory to save evaluation results
        """
        # Load test data
        test_paths, test_labels, _ = self.load_class_data('test')
        
        # Load behavior data
        behavior_data, _ = self.load_behavior_data()
        
        # Create test generator
        test_gen = self.create_data_generator(
            test_paths, test_labels, behavior_data, (224, 224), batch_size=32, augment=False)
        
        # Evaluate model
        print("Evaluating model on test set...")
        evaluation = self.model.evaluate(test_gen)
        
        # Save evaluation metrics
        metrics = {
            'loss': float(evaluation[0]),
            'accuracy': float(evaluation[1])
        }
        
        with open(os.path.join(model_dir, 'test_metrics.json'), 'w') as f:
            json.dump(metrics, f, indent=2)
        
        # Generate predictions
        print("Generating predictions for confusion matrix...")
        y_true = []
        y_pred = []
        
        for i in range(len(test_gen)):
            # Get batch data
            inputs, batch_y = test_gen[i]
            
            # Predict batch
            batch_pred = self.model.predict(inputs)
            
            # Get true and predicted classes
            batch_y_true = np.argmax(batch_y, axis=1)
            batch_y_pred = np.argmax(batch_pred, axis=1)
            
            # Add to lists
            y_true.extend(batch_y_true)
            y_pred.extend(batch_y_pred)
        
        # Create confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        
        # Normalize confusion matrix
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        cm_norm = np.nan_to_num(cm_norm)  # Replace NaN with 0
        
        # Plot confusion matrix
        plt.figure(figsize=(12, 10))
        plt.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title('Normalized Confusion Matrix')
        plt.colorbar()
        
        class_names = self.class_names
        tick_marks = np.arange(len(class_names))
        plt.xticks(tick_marks, class_names, rotation=45)
        plt.yticks(tick_marks, class_names)
        
        # Add text annotations
        fmt = '.2f'
        thresh = cm_norm.max() / 2.
        for i in range(cm_norm.shape[0]):
            for j in range(cm_norm.shape[1]):
                plt.text(j, i, format(cm_norm[i, j], fmt),
                         ha="center", va="center",
                         color="white" if cm_norm[i, j] > thresh else "black")
        
        plt.tight_layout()
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        
        # Save confusion matrix
        plt.savefig(os.path.join(model_dir, 'confusion_matrix.png'), dpi=300)
        plt.close()
        
        # Generate classification report
        report = classification_report(
            y_true, y_pred, target_names=class_names, output_dict=True)
        
        with open(os.path.join(model_dir, 'classification_report.json'), 'w') as f:
            json.dump(report, f, indent=2)
        
        # Print summary
        print(f"Test accuracy: {metrics['accuracy']:.4f}")
        print("Classification report:")
        print(classification_report(y_true, y_pred, target_names=class_names))
    
    def plot_training_history(self, history, output_dir):
        """
        Plot training history
        
        Args:
            history: Training history
            output_dir: Directory to save plots
        """
        # Plot accuracy
        plt.figure(figsize=(10, 6))
        plt.plot(history.history['accuracy'], label='Training')
        plt.plot(history.history['val_accuracy'], label='Validation')
        plt.title('Model Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(output_dir, 'accuracy.png'), dpi=300)
        plt.close()
        
        # Plot loss
        plt.figure(figsize=(10, 6))
        plt.plot(history.history['loss'], label='Training')
        plt.plot(history.history['val_loss'], label='Validation')
        plt.title('Model Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(output_dir, 'loss.png'), dpi=300)
        plt.close()
    
    def predict_image(self, image_path, behavior_features=None):
        """
        Predict emotion for a single image
        
        Args:
            image_path: Path to image file
            behavior_features: Optional list of behavior features
            
        Returns:
            tuple: (predicted_class, confidence, all_predictions)
        """
        if self.model is None:
            raise ValueError("Model not loaded. Train or load a model first.")
        
        # Load and preprocess image
        try:
            img = cv2.imread(image_path)
            if img is None:
                raise ValueError(f"Failed to load image: {image_path}")
            
            # Convert BGR to RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Resize image
            img = cv2.resize(img, (224, 224))
            
            # Normalize pixel values
            img = img.astype(np.float32) / 255.0
            
            # Add batch dimension
            img = np.expand_dims(img, axis=0)
            
            # Create behavior features if not provided
            if behavior_features is None:
                behavior_size = len(self.behavior_columns) if self.behavior_columns else 1
                behavior_features = np.zeros((1, behavior_size), dtype=np.float32)
            else:
                behavior_features = np.array([behavior_features], dtype=np.float32)
            
            # Make prediction
            inputs = {
                'image_input': img,
                'behavior_input': behavior_features
            }
            predictions = self.model.predict(inputs)
            
            # Get predicted class and confidence
            predicted_idx = np.argmax(predictions[0])
            confidence = float(predictions[0][predicted_idx])
            
            if len(self.class_names) > predicted_idx:
                predicted_class = self.class_names[predicted_idx]
            else:
                predicted_class = f"Class {predicted_idx}"
            
            # Create all predictions dict
            all_predictions = {
                self.class_names[i]: float(predictions[0][i]) 
                for i in range(len(self.class_names))
            }
            
            return predicted_class, confidence, all_predictions
        
        except Exception as e:
            print(f"Error predicting image: {str(e)}")
            return None, 0.0, {}
    
    def load_model(self, model_path, metadata_path=None):
        """
        Load a saved model
        
        Args:
            model_path: Path to saved model file (.h5)
            metadata_path: Path to model metadata file (.json)
            
        Returns:
            Loaded model
        """
        try:
            # Load model
            self.model = tf.keras.models.load_model(model_path)
            print(f"Model loaded from {model_path}")
            
            # Load metadata if provided
            if metadata_path and os.path.exists(metadata_path):
                with open(metadata_path, 'r') as f:
                    metadata = json.load(f)
                
                # Extract metadata
                self.class_names = metadata.get('class_names', [])
                self.behavior_columns = metadata.get('behavior_columns', [])
                
                print(f"Loaded metadata: {len(self.class_names)} classes, {len(self.behavior_columns)} behavior features")
            
            return self.model
        
        except Exception as e:
            print(f"Error loading model: {str(e)}")
            return None

# Helper function to find the train_by_class directory
def find_directory(base_dir="/content/drive/MyDrive/Colab Notebooks/Pawnder", target_dir="train_by_class"):
    """
    Find a directory recursively starting from the base_dir
    
    Args:
        base_dir: Directory to start the search
        target_dir: Directory name to find
        
    Returns:
        Path to the found directory or None
    """
    print(f"Searching for {target_dir} in {base_dir}")
    
    for root, dirs, files in os.walk(base_dir):
        if target_dir in dirs:
            found_dir = os.path.join(root, target_dir)
            print(f"Found {target_dir} at {found_dir}")
            return found_dir
    
    return None

# Usage example
if __name__ == "__main__":
    # Try to find the correct directory
    train_dir = find_directory()
    if train_dir:
        # Use the parent directory as the processed_dir
        processed_dir = os.path.dirname(train_dir)
        print(f"Using {processed_dir} as the processed directory")
        
        # Create classifier with the found directory
        classifier = DogEmotionWithBehaviors(base_dir=os.path.dirname(processed_dir))
    else:
        # Use default directory
        classifier = DogEmotionWithBehaviors()
    
    # Train model
    history, model_dir = classifier.train(
        epochs=50,
        batch_size=32,
        fine_tune=True
    )
    
    print(f"Training completed. Model saved to {model_dir}")
    f.write("""
# Code content is in the artifact above
    """)

# Part 6: Main Execution

In [ ]:
def main(args=None):
    """Main execution function"""
    import argparse
    
    # Parse command line arguments if provided
    if args is None:
        parser = argparse.ArgumentParser(description='Dog Emotion Recognition Pipeline')
        parser.add_argument('--process_data', action='store_true', help='Process the dataset')
        parser.add_argument('--train', action='store_true', help='Train the model')
        parser.add_argument('--evaluate', action='store_true', help='Evaluate the model')
        parser.add_argument('--predict', type=str, help='Predict emotion for an image file')
        parser.add_argument('--process_video', type=str, help='Process a video file')
        parser.add_argument('--model_type', type=str, default='image', choices=['image', 'video', 'region'], 
                            help='Model type to use')
        parser.add_argument('--model_path', type=str, help='Path to a saved model for prediction/evaluation')
        parser.add_argument('--output', type=str, help='Output path for video processing')
        args = parser.parse_args()
    
    # Make sure at least one action is specified
    if not any([args.process_data, args.train, args.evaluate, args.predict, args.process_video]):
        print("No action specified. Please use --process_data, --train, --evaluate, --predict, or --process_video")
        return
    
    # Process data if requested
    if args.process_data:
        all_frames, split_counts = process_data()
        print("Data processing complete!")
    
    # Train model if requested
    if args.train:
        print(f"Training a {args.model_type} model...")
        
        # Create model builder
        model_builder = DogEmotionModel()
        
        # Create model based on specified type
        if args.model_type == 'region':
            model = model_builder.create_region_attention_model()
        elif args.model_type == 'video':
            model = model_builder.create_model(model_type='video')
        else:
            model = model_builder.create_model(model_type='image')
        
        # Train the model
        model, history = train_model(
            model, 
            DATA_DIRS['processed'], 
            config, 
            model_name=f"dog_emotion_{args.model_type}"
        )
        
        # Save model path for evaluation or prediction
        args.model_path = os.path.join(
            config['training']['checkpoint_dir'],
            f"dog_emotion_{args.model_type}_final.h5"
        )
        
        print(f"Training complete! Model saved to {args.model_path}")
    
    # Evaluate model if requested
    if args.evaluate:
        # Make sure we have a model
        if args.model_path is None and 'model' not in locals():
            print("No model available for evaluation. Please specify --model_path or run --train first.")
            return
        
        # Load model if needed
        if 'model' not in locals():
            print(f"Loading model from {args.model_path}")
            model = tf.keras.models.load_model(args.model_path)
        
        # Evaluate the model
        evaluation = evaluate_model(
            model, 
            DATA_DIRS['processed'], 
            config, 
            model_name=os.path.basename(args.model_path).split('.')[0]
        )
        
        print("Evaluation complete!")
    
    # Predict for a single image if requested
    if args.predict:
        # Make sure we have a model
        if args.model_path is None and 'model' not in locals():
            print("No model available for prediction. Please specify --model_path or run --train first.")
            return
        
        # Load model if needed
        if 'model' not in locals():
            print(f"Loading model from {args.model_path}")
            model = tf.keras.models.load_model(args.model_path)
        
        # Make prediction
        result = predict_image(model, args.predict, config)
        
        print(f"\nPrediction for {args.predict}:")
        print(f"Emotion: {result['emotion']} (Score: {result['emotion_score']:.2f})")
        print(f"Confidence: {result['confidence']:.2f}")
        
        print("\nTop 3 emotions:")
        for emotion, score in sorted(result['all_emotions'].items(), key=lambda x: x[1], reverse=True)[:3]:
            print(f"- {emotion}: {score:.2f}")
    
    # Process video if requested
    if args.process_video:
        # Make sure we have a model
        if args.model_path is None and 'model' not in locals():
            print("No model available for video processing. Please specify --model_path or run --train first.")
            return
        
        # Load model if needed
        if 'model' not in locals():
            print(f"Loading model from {args.model_path}")
            model = tf.keras.models.load_model(args.model_path)
        
        # Determine output path
        output_path = args.output
        if output_path is None:
            # Create default output path
            base_name = os.path.basename(args.process_video)
            output_dir = os.path.join(DATA_DIRS['processed'], "predictions")
            os.makedirs(output_dir, exist_ok=True)
            output_path = os.path.join(output_dir, f"analyzed_{base_name}")
        
        # Process video
        results = process_video(model, args.process_video, config, output_path=output_path)
        
        print(f"\nVideo Analysis Results for {args.process_video}")
        print(f"Dominant Emotion: {results['dominant_emotion']}")
        print("\nEmotion Distribution:")
        for emotion, percent in sorted(
            results['emotion_distribution'].items(), 
            key=lambda x: x[1], 
            reverse=True
        ):
            print(f"  {emotion}: {percent:.1%}")
        
        print(f"\nProcessed video saved to: {output_path}")

if __name__ == "__main__":
    main()